# multivon-eval — Quickstart

[![PyPI](https://img.shields.io/pypi/v/multivon-eval.svg)](https://pypi.org/project/multivon-eval)
[![Docs](https://img.shields.io/badge/docs-evaldocs.multivon.ai-violet)](https://evaldocs.multivon.ai)

This notebook walks through the core API in ~5 minutes.

**Parts 1–3 run with no API key** (deterministic evaluators).  
**Part 4** shows LLM-judge evaluators — drop in your `ANTHROPIC_API_KEY` or `OPENAI_API_KEY` to run that section.

Docs: [evaldocs.multivon.ai](https://evaldocs.multivon.ai) · GitHub: [multivon-ai/multivon-eval](https://github.com/multivon-ai/multivon-eval)

In [ ]:
!pip install multivon-eval -q

## Part 1 — Deterministic evaluators (no API key needed)

Deterministic evaluators are instant, free, and need no LLM call.  
Good for: format checks, exact match, keyword presence, length, latency.

In [ ]:
from multivon_eval import EvalSuite, EvalCase, NotEmpty, ExactMatch, Contains, WordCount

# A simple model function — swap this for your real LLM call
def my_model(input: str) -> str:
    answers = {
        "What is the capital of France?": "Paris",
        "What is 2 + 2?": "4",
        "Name a planet.": "Mars is a planet in our solar system.",
    }
    return answers.get(input, "I don't know.")

suite = EvalSuite("Deterministic Demo")

suite.add_cases([
    EvalCase(input="What is the capital of France?", expected_output="Paris"),
    EvalCase(input="What is 2 + 2?", expected_output="4"),
    EvalCase(input="Name a planet."),
])

suite.add_evaluators(
    NotEmpty(),            # output is not blank
    ExactMatch(),          # matches expected_output exactly (where set)
    Contains(["planet"]),  # output contains keyword
    WordCount(min=1, max=50),  # output length in range
)

report = suite.run(my_model)

In [ ]:
# The report object is fully inspectable
print(f"Pass rate:  {report.pass_rate:.0%}")
print(f"Avg score:  {report.avg_score:.2f}")
print(f"Cases run:  {len(report.case_results)}")

## Part 2 — BLEU and ROUGE (no API key needed)

Reference-based metrics for summarization and translation tasks.

In [ ]:
from multivon_eval import BLEU, ROUGE

def summarizer(text: str) -> str:
    # Toy summarizer — replace with your model
    return text.split(".")[0] + "."

suite2 = EvalSuite("Summarization")
suite2.add_cases([
    EvalCase(
        input="Summarize: Climate change is accelerating. Polar ice is melting rapidly.",
        expected_output="Climate change is accelerating.",
    ),
])
suite2.add_evaluators(NotEmpty(), BLEU(), ROUGE())
report2 = suite2.run(summarizer)

## Part 3 — Multi-run + flakiness detection (no API key needed)

LLMs are non-deterministic. Run each case multiple times to catch flaky outputs.
Backed by [NAACL 2025](https://arxiv.org/abs/2502.01775): single-run eval scores are unreliable.

In [ ]:
import random

def flaky_model(input: str) -> str:
    # Simulates a non-deterministic model
    return "Paris" if random.random() > 0.4 else ""

suite3 = EvalSuite("Flakiness Demo")
suite3.add_cases([EvalCase(input="Capital of France?", expected_output="Paris")])
suite3.add_evaluators(NotEmpty(), ExactMatch())

# Run each case 5 times — flaky cases are flagged in the report
report3 = suite3.run(flaky_model, runs=5)

## Part 4 — LLM-judge evaluators (API key required)

Set your key below. multivon-eval supports Anthropic and OpenAI out of the box.

In [ ]:
import os

# Set ONE of these:
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # claude-haiku-4-5 by default
# os.environ["OPENAI_API_KEY"] = "sk-..."         # gpt-4o-mini by default

In [ ]:
from multivon_eval import Faithfulness, Relevance, Hallucination

def support_bot(question: str) -> str:
    # Replace with your real model call
    return "You can reset your password by clicking 'Forgot Password' on the login page."

suite4 = EvalSuite("Support Bot — LLM Judge")
suite4.add_cases([
    EvalCase(
        input="How do I reset my password?",
        context="Users can reset their password by clicking 'Forgot Password' on the login page.",
    ),
])
suite4.add_evaluators(
    NotEmpty(),
    Faithfulness(),    # does the answer stay within the provided context?
    Relevance(),       # is the answer relevant to the question?
    Hallucination(),   # does the answer introduce claims not in the context?
)
report4 = suite4.run(support_bot)

## Part 5 — Factory suites (one line setup)

Pre-built suites with the right evaluators for common use cases.

In [ ]:
from multivon_eval import EvalSuite

# One line — right evaluators, sensible defaults
suite = EvalSuite.for_rag()          # Faithfulness, Hallucination, ContextPrecision, ContextRecall, Relevance
# suite = EvalSuite.for_support_bot()  # Faithfulness, Relevance, Coherence, Toxicity
# suite = EvalSuite.for_classification()  # NotEmpty, ExactMatch, AnswerAccuracy

print("Available factory suites:")
print("  EvalSuite.for_rag()")
print("  EvalSuite.for_agents()")
print("  EvalSuite.for_support_bot()")
print("  EvalSuite.for_summarization()")
print("  EvalSuite.for_chatbot()")
print("  EvalSuite.for_classification()")
print("  EvalSuite.for_document_intelligence(schema=MyModel)")
print("  EvalSuite.for_regulated(jurisdiction='hipaa')")

## Next steps

- **Docs:** [evaldocs.multivon.ai](https://evaldocs.multivon.ai)
- **Agent evaluators:** tool call accuracy, trajectory efficiency, plan quality → [Agent Evaluators](https://evaldocs.multivon.ai/evaluators/agent)
- **Experiment tracking:** compare runs with p-values → [Experiment Tracking](https://evaldocs.multivon.ai/guides/experiments)
- **CI/CD integration:** block deploys on regression → [CI/CD Guide](https://evaldocs.multivon.ai/guides/ci-cd)
- **Compliance:** HIPAA PII detection, audit trails → [Compliance Guide](https://evaldocs.multivon.ai/guides/compliance)
- **GitHub:** [github.com/multivon-ai/multivon-eval](https://github.com/multivon-ai/multivon-eval)